# Small Model Learnability Gap in the Code Domain — Thesis Notebook

This notebook investigates whether the **Small Model Learnability Gap** (Li et al. 2025) — originally observed in math reasoning — also appears in code generation, and tests whether a **dynamic curriculum mix** outperforms the paper's static Mix-Long recipe.

## Research design

**Student model:** `Qwen/Qwen2.5-1.5B-Instruct` (general instruct, not coder-specialized — matches paper's setup; isolates the learnability question from domain pre-training effects).

**Training conditions (5 total):**
1. Base — no fine-tuning (reference)
2. Long-CoT only
3. Short-CoT only
4. Static Mix-Long (α=0.2) — paper's exact recipe
5. Curriculum Mix — dynamic α schedule across 3 stages

**Evaluation:**
- **MBPP held-out test set (task_ids 11–510)** — *in-domain* generalization
- **HumanEval (164 problems)** — *out-of-distribution* transfer
- Greedy decoding for deterministic Pass@1
- Per-stage evaluation for the curriculum to track learning trajectory

**Diagnostics:**
- Train/test overlap audit: scans curriculum data for any leakage into MBPP test.
- Perplexity analysis: replicates paper's Appendix B.3 in the code domain.


## 1. Environment setup (RunPod)

**Recommended pod spec:** any GPU with ≥16 GB VRAM. The original Kaggle T4 (16 GB) works; an A4000/A5000/RTX 4090/A100 will be 2–5× faster and lets you raise `PER_DEVICE_BATCH` if you want.

**Persistent volume:** mount your network volume at `/workspace` (this is the RunPod default). All adapters, eval results, and logs are written there so they survive pod restarts.

**Before running:** make sure your curriculum JSONL files are placed at `CURRICULUM_DIR` (see config cell). Cell 3a below shows three ways to get them onto the pod — pick whichever fits.


In [ ]:
# Clear any previous unsloth compiled cache (RunPod persistent volume)
!rm -rf /workspace/unsloth_compiled_cache
!rm -rf ~/.cache/unsloth_compiled_cache


In [ ]:
# Install dependencies for RunPod
# Most RunPod PyTorch templates ship with torch + CUDA already installed,
# so we install Unsloth without forcing a torch reinstall.
!pip install -q --upgrade pip
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade trl transformers datasets accelerate peft bitsandbytes
!pip install -q human-eval evalplus tensorboard

# Quick GPU sanity check
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 support:   {torch.cuda.is_bf16_supported()}")


In [ ]:
# === Dataset acquisition (run ONE of the three options below) ===
# The notebook expects three JSONL files at CURRICULUM_DIR (set in the config cell):
#   epoch_1.jsonl, epoch_2.jsonl, epoch_3.jsonl
#
# Pick the option that matches how you stored your dataset:

import os
os.makedirs("/workspace/datasets/curriculum", exist_ok=True)

# --- OPTION A: files already uploaded manually to /workspace/datasets/curriculum/ ---
# (e.g. via `runpodctl send`, scp, or the RunPod web file manager)
# Nothing to do — just verify the files exist:
for fname in ["epoch_1.jsonl", "epoch_2.jsonl", "epoch_3.jsonl"]:
    path = f"/workspace/datasets/curriculum/{fname}"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  ✓ {path}  ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ MISSING: {path}")

# --- OPTION B: pull from a HuggingFace dataset repo ---
# Uncomment and set HF_DATASET_REPO if you pushed your curriculum to HF Hub.
# from huggingface_hub import snapshot_download
# HF_DATASET_REPO = "haris77777/thesis-dynamic-mix-curriculum"   # change me
# snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
#                   local_dir="/workspace/datasets/curriculum")

# --- OPTION C: pull from Kaggle datasets (needs ~/.kaggle/kaggle.json) ---
# !pip install -q kaggle
# !mkdir -p ~/.kaggle
# # Upload your kaggle.json to the pod, then:
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d haris77777/thesis-dynamic-mix-curriculum \
#     -p /workspace/datasets/curriculum --unzip


In [ ]:
# Verify dataset files on /workspace
import os

DATA_ROOT = "/workspace/datasets"
print(f"=== {DATA_ROOT}/ contents ===")
if os.path.exists(DATA_ROOT):
    for item in os.listdir(DATA_ROOT):
        path = os.path.join(DATA_ROOT, item)
        if os.path.isdir(path):
            print(f"  {item}/")
            for sub in os.listdir(path):
                sub_path = os.path.join(path, sub)
                size_mb = os.path.getsize(sub_path) / 1024 / 1024 if os.path.isfile(sub_path) else 0
                print(f"    {sub}  ({size_mb:.1f} MB)")
else:
    print(f"  (directory does not exist yet — run the dataset acquisition cell above)")

print("\n=== JSONL files found anywhere under /workspace/datasets ===")
for root, _, files in os.walk(DATA_ROOT):
    for f in files:
        if f.endswith('.jsonl'):
            path = os.path.join(root, f)
            size_mb = os.path.getsize(path) / 1024 / 1024
            print(f"  {path}  ({size_mb:.1f} MB)")


In [ ]:
# Global config — change paths here if your dataset lives elsewhere
import os

# RunPod paths: /workspace is the persistent network-volume mount
CURRICULUM_DIR = "/workspace/datasets/curriculum"
EPOCH_FILES = [
    f"{CURRICULUM_DIR}/epoch_1.jsonl",
    f"{CURRICULUM_DIR}/epoch_2.jsonl",
    f"{CURRICULUM_DIR}/epoch_3.jsonl",
]

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 4096          # was 2048 — paper's avg long-CoT ≈ 3.4k tokens
SEED = 3407

# LoRA config
LORA_R = 32                    # was 16 — bigger adapter for reasoning capacity
LORA_ALPHA = 64                # was 16 — typically 2*r
LORA_DROPOUT = 0.0

# Training
PER_DEVICE_BATCH = 2           # raise to 4 or 8 if you're on an A100/4090 with VRAM headroom
GRAD_ACCUM = 4                 # effective batch = PER_DEVICE_BATCH * GRAD_ACCUM
LEARNING_RATE = 2e-4
NUM_EPOCHS_PER_RUN = 2         # paper uses 2 epochs

# Eval
EVAL_GREEDY = True             # deterministic, paper-comparable
EVAL_MAX_NEW_TOKENS = 1024

# MBPP held-out test split (Austin et al. standard)
MBPP_TEST_TASK_IDS = list(range(11, 511))   # 500 problems

# Output dirs — all under /workspace so they persist across pod restarts
OUTPUT_ROOT = "/workspace/outputs"
os.makedirs(f"{OUTPUT_ROOT}/adapters", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/eval_results", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/logs", exist_ok=True)

# Optional: set HF cache to /workspace too, so you don't re-download Qwen on every pod restart
os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("TRANSFORMERS_CACHE", "/workspace/.cache/huggingface/hub")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# If you want to use a gated model or push results to HF Hub, uncomment and set:
# os.environ["HF_TOKEN"] = "hf_..."   # or set as a RunPod env var instead

print(f"Base model:        {BASE_MODEL}")
print(f"Max seq length:    {MAX_SEQ_LENGTH}")
print(f"LoRA r/alpha:      {LORA_R}/{LORA_ALPHA}")
print(f"Effective batch:   {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"MBPP test size:    {len(MBPP_TEST_TASK_IDS)}")
print(f"Output root:       {OUTPUT_ROOT}")
print(f"HF cache:          {os.environ['HF_HOME']}")


In [ ]:
# Set seeds for reproducibility
import random, numpy as np, torch
from transformers import set_seed

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print(f"All seeds set to {SEED}")

## 2. Data inspection & train/test overlap audit

This section addresses **train/test domain coupling**. We:
1. Inspect curriculum file structure.
2. Load MBPP held-out test split (task_ids 11–510).
3. Check for any overlap by task_id and by prompt text similarity.

If any overlap is found, we'll see it here *before* training.

In [ ]:
import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

# Inspect the first epoch file
sample = load_jsonl(EPOCH_FILES[0])
print(f"Epoch 1 has {len(sample)} examples")
print(f"\nFields in first example: {list(sample[0].keys())}")
print(f"\nFirst example (truncated):")
for k, v in sample[0].items():
    s = str(v)
    print(f"  {k}: {s[:200]}{'...' if len(s) > 200 else ''}")

In [ ]:
# Count long_cot vs short_cot availability in each epoch
for i, path in enumerate(EPOCH_FILES, 1):
    data = load_jsonl(path)
    n_long = sum(1 for ex in data if ex.get('long_cot'))
    n_short = sum(1 for ex in data if ex.get('short_cot'))
    n_both = sum(1 for ex in data if ex.get('long_cot') and ex.get('short_cot'))
    print(f"Epoch {i}: total={len(data)}  long={n_long}  short={n_short}  both={n_both}")

In [ ]:
# Load MBPP held-out test set
from datasets import load_dataset

mbpp_full = load_dataset("mbpp", "full", split="test")
# The "test" split in HF mbpp corresponds to task_ids 11-510 already
mbpp_test = [ex for ex in mbpp_full if ex['task_id'] in MBPP_TEST_TASK_IDS]
print(f"MBPP test problems: {len(mbpp_test)}")
print(f"Example: task_id={mbpp_test[0]['task_id']}")
print(f"  text: {mbpp_test[0]['text'][:150]}")
print(f"  test_list: {mbpp_test[0]['test_list']}")

In [ ]:
# === Train/Test overlap audit ===
# Check 1: any explicit task_id overlap
# Check 2: any prompt text overlap (substring match on normalized text)

import re

def normalize(s):
    return re.sub(r'\s+', ' ', s.lower().strip())[:200] if s else ''

mbpp_test_ids = set(ex['task_id'] for ex in mbpp_test)
mbpp_test_prompts = set(normalize(ex['text']) for ex in mbpp_test)

print("=== Overlap audit ===")
total_train = 0
id_overlap = 0
text_overlap = []

for path in EPOCH_FILES:
    data = load_jsonl(path)
    total_train += len(data)
    for ex in data:
        # task_id check (handle both 'task_id' and 'id' field names)
        tid = ex.get('task_id') or ex.get('id')
        if tid and tid in mbpp_test_ids:
            id_overlap += 1
        # text-prompt check
        prompt_norm = normalize(ex.get('prompt', ''))
        if prompt_norm and prompt_norm in mbpp_test_prompts:
            text_overlap.append(tid)

print(f"Total training examples:      {total_train}")
print(f"task_id overlap with MBPP test: {id_overlap}")
print(f"Prompt-text overlap:           {len(text_overlap)}")

if id_overlap > 0 or len(text_overlap) > 0:
    print("\n⚠️  LEAKAGE DETECTED — inspect before training!")
else:
    print("\n✓ No overlap detected. MBPP test is genuinely held out.")

## 3. Tokenizer & data formatting

Switching from Alpaca template to Qwen's native ChatML via `tokenizer.apply_chat_template()`.
This preserves the model's instruction-following alignment.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

SYSTEM_PROMPT = "You are an expert Python programmer. Solve the given task and return correct, working code."

def to_chat_text(prompt, response):
    """Apply Qwen's ChatML template — used for both training and eval."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

def to_chat_prompt(prompt):
    """Same template but without assistant content — for inference."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Sanity-check the template
print(to_chat_text("Write a function that returns the square of n.", "def square(n):\n    return n * n"))

In [ ]:
from datasets import Dataset

def build_dataset(records, mode):
    """
    mode in {'long', 'short', 'mix_static', 'curriculum_native'}
      - 'long':              use long_cot only (skip rows missing it)
      - 'short':             use short_cot only
      - 'mix_static':        20% long, 80% short (paper's α=0.2)
      - 'curriculum_native': use long if present, else short (original behavior)
    """
    rng = random.Random(SEED)
    rows = []

    if mode == 'long':
        for ex in records:
            if ex.get('long_cot'):
                rows.append({'text': to_chat_text(ex['prompt'], ex['long_cot'])})

    elif mode == 'short':
        for ex in records:
            if ex.get('short_cot'):
                rows.append({'text': to_chat_text(ex['prompt'], ex['short_cot'])})

    elif mode == 'mix_static':
        # α=0.2 long, 0.8 short
        for ex in records:
            has_long = bool(ex.get('long_cot'))
            has_short = bool(ex.get('short_cot'))
            if has_long and has_short:
                resp = ex['long_cot'] if rng.random() < 0.2 else ex['short_cot']
            elif has_long:
                resp = ex['long_cot']
            elif has_short:
                resp = ex['short_cot']
            else:
                continue
            rows.append({'text': to_chat_text(ex['prompt'], resp)})

    elif mode == 'curriculum_native':
        for ex in records:
            resp = ex.get('long_cot') or ex.get('short_cot')
            if resp:
                rows.append({'text': to_chat_text(ex['prompt'], resp)})

    rng.shuffle(rows)
    return Dataset.from_list(rows)

# Aggregate all curriculum files for the non-curriculum runs
all_records = []
for path in EPOCH_FILES:
    all_records.extend(load_jsonl(path))
print(f"Total aggregated records: {len(all_records)}")

## 4. Evaluation helpers

Two benchmarks:
- **MBPP held-out test**: in-domain generalization (500 problems, task_ids 11–510)
- **HumanEval**: OOD transfer (164 problems)

Both use **greedy decoding** for deterministic, paper-comparable Pass@1.
Pass@1 confidence intervals via Wilson score interval.

In [ ]:
import re, signal, subprocess, tempfile, json
from tqdm.auto import tqdm

def extract_code(text):
    """Extract Python code from a model response. Tries fenced blocks first."""
    fenced = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if fenced:
        return fenced[-1].strip()
    return text.strip()

def wilson_interval(successes, total, z=1.96):
    """95% Wilson confidence interval for a proportion."""
    if total == 0:
        return (0.0, 0.0)
    p = successes / total
    denom = 1 + z**2 / total
    centre = (p + z**2 / (2*total)) / denom
    half = (z * ((p*(1-p)/total + z**2/(4*total**2))**0.5)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))

In [ ]:
# Sandboxed code execution helper
import multiprocessing as mp

def _run_in_subprocess(code_str, timeout=10):
    """Run code in a subprocess with a timeout. Returns True if it completes without error."""
    def target(q, code_str):
        try:
            exec_globals = {'__name__': '__main__'}
            exec(code_str, exec_globals)
            q.put(('ok', None))
        except Exception as e:
            q.put(('err', f"{type(e).__name__}: {e}"))

    ctx = mp.get_context('fork')
    q = ctx.Queue()
    p = ctx.Process(target=target, args=(q, code_str))
    p.start()
    p.join(timeout)
    if p.is_alive():
        p.terminate()
        p.join(1)
        return False, 'timeout'
    if not q.empty():
        status, msg = q.get()
        return status == 'ok', msg
    return False, 'no_result'

def check_mbpp(completion, test_list, setup_code=''):
    """Run MBPP test_list against the completion."""
    full = (setup_code + '\n' + completion + '\n' + '\n'.join(test_list))
    return _run_in_subprocess(full, timeout=10)

def check_humaneval(completion, test_code, entry_point):
    """Run HumanEval test against the completion."""
    full = completion + '\n' + test_code + f'\ncheck({entry_point})'
    return _run_in_subprocess(full, timeout=10)

In [ ]:
def generate_completion(model, tokenizer, user_prompt, max_new_tokens=EVAL_MAX_NEW_TOKENS):
    """Generate a single completion using greedy decoding."""
    text = to_chat_prompt(user_prompt)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,            # greedy
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return full

def eval_humaneval(model, tokenizer, label):
    """Run HumanEval Pass@1 with greedy decoding."""
    he = load_dataset("openai_humaneval", split="test")
    n_pass = 0
    results = []
    for problem in tqdm(he, desc=f"HumanEval [{label}]"):
        prompt = (
            f"Complete the following Python function. "
            f"Return only the full function in a python code block.\n\n"
            f"```python\n{problem['prompt']}\n```"
        )
        response = generate_completion(model, tokenizer, prompt)
        code_str = extract_code(response)
        # Some completions are just the body; if the def is missing, prepend the prompt
        if 'def ' not in code_str:
            code_str = problem['prompt'] + '\n' + code_str
        ok, msg = check_humaneval(code_str, problem['test'], problem['entry_point'])
        if ok:
            n_pass += 1
        results.append({
            'task_id': problem['task_id'],
            'passed': ok,
            'error': msg if not ok else None,
            'completion': code_str,
        })
    pass_rate = n_pass / len(he)
    lo, hi = wilson_interval(n_pass, len(he))
    return {
        'benchmark': 'humaneval',
        'label': label,
        'pass@1': pass_rate,
        'n_pass': n_pass,
        'n_total': len(he),
        'ci95': [lo, hi],
        'results': results,
    }

def eval_mbpp(model, tokenizer, label):
    """Run MBPP held-out (task_ids 11-510) Pass@1 with greedy decoding."""
    n_pass = 0
    results = []
    for problem in tqdm(mbpp_test, desc=f"MBPP-test [{label}]"):
        # Show first test as a hint (standard MBPP eval protocol)
        first_test = problem['test_list'][0]
        user_prompt = (
            f"You are given a programming task. Write a Python function that solves it.\n\n"
            f"Task: {problem['text']}\n\n"
            f"Your code must pass this test: {first_test}\n\n"
            f"Return only the function in a python code block."
        )
        response = generate_completion(model, tokenizer, user_prompt)
        code_str = extract_code(response)
        ok, msg = check_mbpp(code_str, problem['test_list'], problem.get('test_setup_code', ''))
        if ok:
            n_pass += 1
        results.append({
            'task_id': problem['task_id'],
            'passed': ok,
            'error': msg if not ok else None,
            'completion': code_str,
        })
    pass_rate = n_pass / len(mbpp_test)
    lo, hi = wilson_interval(n_pass, len(mbpp_test))
    return {
        'benchmark': 'mbpp_test',
        'label': label,
        'pass@1': pass_rate,
        'n_pass': n_pass,
        'n_total': len(mbpp_test),
        'ci95': [lo, hi],
        'results': results,
    }

def save_eval(result):
    path = f"{OUTPUT_ROOT}/eval_results/{result['label']}__{result['benchmark']}.json"
    with open(path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f"  saved -> {path}")
    return path

def report(result):
    lo, hi = result['ci95']
    print(f"  {result['benchmark']:12s} | {result['label']:25s} | "
          f"Pass@1 = {result['pass@1']*100:5.2f}% "
          f"({result['n_pass']}/{result['n_total']})  "
          f"95% CI [{lo*100:.2f}, {hi*100:.2f}]")

## 5. Training driver

Each training condition gets its own cell. After training, the LoRA adapter is saved and the model is freed before the next run.

**Compute budget:** ~30–60 min per training run on a T4. Run cells in order; if you hit a session limit, you can resume from any saved adapter.

In [ ]:
import gc
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def load_fresh_base():
    """Load a fresh base model + LoRA adapter for a new training run."""
    free_gpu()
    model, tok = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
    return model, tok

def run_sft(model, tok, dataset, output_subdir, num_epochs=NUM_EPOCHS_PER_RUN):
    out_dir = f"{OUTPUT_ROOT}/logs/{output_subdir}"
    trainer = SFTTrainer(
        model=model,
        processing_class=tok,
        train_dataset=dataset,
        args=SFTConfig(
            dataset_text_field='text',
            max_seq_length=MAX_SEQ_LENGTH,
            dataset_num_proc=2,
            padding_free=False,
            per_device_train_batch_size=PER_DEVICE_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM,
            warmup_steps=5,
            num_train_epochs=num_epochs,
            learning_rate=LEARNING_RATE,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            output_dir=out_dir,
            report_to="tensorboard",
            seed=SEED,
        ),
    )
    return trainer.train()

print("Training driver ready.")

## 6. Training runs

Run these cells in any order. Each saves its adapter independently.

**Recommended priority if compute is tight:**
`static_mix_alpha02` → `curriculum` → `long_only` → `short_only`

### 6.1 Long-CoT only

In [ ]:
# === Train: Long-CoT only ===
ADAPTER_PATH = f"{OUTPUT_ROOT}/adapters/long_only"

dataset = build_dataset(all_records, mode='long')
print(f"Long-only dataset size: {len(dataset)}")

model, tok = load_fresh_base()
run_sft(model, tok, dataset, output_subdir="long_only")

model.save_pretrained(ADAPTER_PATH)
tok.save_pretrained(ADAPTER_PATH)
print(f"Saved adapter to {ADAPTER_PATH}")

del model, tok, dataset
free_gpu()

### 6.2 Short-CoT only

In [ ]:
# === Train: Short-CoT only ===
ADAPTER_PATH = f"{OUTPUT_ROOT}/adapters/short_only"

dataset = build_dataset(all_records, mode='short')
print(f"Short-only dataset size: {len(dataset)}")

model, tok = load_fresh_base()
run_sft(model, tok, dataset, output_subdir="short_only")

model.save_pretrained(ADAPTER_PATH)
tok.save_pretrained(ADAPTER_PATH)
print(f"Saved adapter to {ADAPTER_PATH}")

del model, tok, dataset
free_gpu()

### 6.3 Static Mix-Long (α=0.2) — paper's recipe

This is the **headline baseline** for your curriculum-vs-static comparison.

In [ ]:
# === Train: Static Mix-Long (α=0.2) ===
ADAPTER_PATH = f"{OUTPUT_ROOT}/adapters/static_mix_alpha02"

dataset = build_dataset(all_records, mode='mix_static')
print(f"Static-mix dataset size: {len(dataset)}")

model, tok = load_fresh_base()
run_sft(model, tok, dataset, output_subdir="static_mix")

model.save_pretrained(ADAPTER_PATH)
tok.save_pretrained(ADAPTER_PATH)
print(f"Saved adapter to {ADAPTER_PATH}")

del model, tok, dataset
free_gpu()

### 6.4 Curriculum Mix (3 stages, with per-stage evaluation)

This is your **novel contribution**. Per-stage Pass@1 is recorded so you can plot the learning trajectory.

In [ ]:
# === Train: Curriculum Mix (3 stages) ===
# Per-stage evaluation tracks how Pass@1 evolves across the curriculum.

curriculum_trajectory = []

model, tok = load_fresh_base()

for stage_idx, path in enumerate(EPOCH_FILES, 1):
    print(f"\n{'='*60}\n  CURRICULUM STAGE {stage_idx}\n{'='*60}")

    records = load_jsonl(path)
    dataset = build_dataset(records, mode='curriculum_native')
    print(f"Stage {stage_idx} dataset size: {len(dataset)}")

    # Train ONE epoch per stage (3 stages × 1 epoch ≈ 3 epochs total, comparable to other runs)
    run_sft(model, tok, dataset, output_subdir=f"curriculum_stage{stage_idx}", num_epochs=1)

    stage_path = f"{OUTPUT_ROOT}/adapters/curriculum_stage{stage_idx}"
    model.save_pretrained(stage_path)
    tok.save_pretrained(stage_path)

    # Per-stage eval — switch model to inference mode briefly
    FastLanguageModel.for_inference(model)
    he_res = eval_humaneval(model, tok, label=f"curriculum_stage{stage_idx}")
    save_eval(he_res); report(he_res)
    mbpp_res = eval_mbpp(model, tok, label=f"curriculum_stage{stage_idx}")
    save_eval(mbpp_res); report(mbpp_res)

    curriculum_trajectory.append({
        'stage': stage_idx,
        'humaneval_pass1': he_res['pass@1'],
        'mbpp_test_pass1': mbpp_res['pass@1'],
    })

    # Switch back to training mode for next stage
    FastLanguageModel.for_training(model)

# Final adapter is the same as stage 3
import shutil
final_path = f"{OUTPUT_ROOT}/adapters/curriculum_final"
if os.path.exists(final_path):
    shutil.rmtree(final_path)
shutil.copytree(f"{OUTPUT_ROOT}/adapters/curriculum_stage3", final_path)

with open(f"{OUTPUT_ROOT}/eval_results/curriculum_trajectory.json", "w") as f:
    json.dump(curriculum_trajectory, f, indent=2)

print("\n=== Curriculum trajectory ===")
for s in curriculum_trajectory:
    print(f"  Stage {s['stage']}: HumanEval {s['humaneval_pass1']*100:.2f}%  |  MBPP-test {s['mbpp_test_pass1']*100:.2f}%")

del model, tok
free_gpu()

## 7. Final evaluation: all conditions on both benchmarks

For each condition we report Pass@1 with 95% Wilson CIs on both MBPP-test (in-domain) and HumanEval (OOD).

In [ ]:
# Helper to load adapter + run both benchmarks
def eval_condition(label, adapter_path=None):
    """Evaluate a single condition. If adapter_path is None, evaluates the base model."""
    print(f"\n=== Evaluating: {label} ===")
    free_gpu()

    model_name = adapter_path if adapter_path else BASE_MODEL
    model, tok = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    he_res = eval_humaneval(model, tok, label=label)
    save_eval(he_res); report(he_res)

    mbpp_res = eval_mbpp(model, tok, label=label)
    save_eval(mbpp_res); report(mbpp_res)

    del model, tok
    free_gpu()
    return he_res, mbpp_res

In [ ]:
# Base model — reference point
base_he, base_mbpp = eval_condition("base", adapter_path=None)

In [ ]:
# Long-CoT only
long_he, long_mbpp = eval_condition("long_only", adapter_path=f"{OUTPUT_ROOT}/adapters/long_only")

In [ ]:
# Short-CoT only
short_he, short_mbpp = eval_condition("short_only", adapter_path=f"{OUTPUT_ROOT}/adapters/short_only")

In [ ]:
# Static Mix (paper's recipe)
mix_he, mix_mbpp = eval_condition("static_mix_alpha02", adapter_path=f"{OUTPUT_ROOT}/adapters/static_mix_alpha02")

In [ ]:
# Curriculum (final stage already evaluated above; load and re-run for clean results.json file)
curr_he, curr_mbpp = eval_condition("curriculum_final", adapter_path=f"{OUTPUT_ROOT}/adapters/curriculum_final")

## 8. Distribution diagnostics — perplexity analysis

Replicates paper's Appendix B.3 in the code domain. We measure the perplexity that the **base** Qwen2.5-1.5B-Instruct assigns to long-CoT vs short-CoT training responses.

If long-CoT PPL > short-CoT PPL by a wide margin, that's empirical evidence of the distribution mismatch hypothesis (Takeaway 4) for code reasoning.

In [ ]:
# Sample 200 examples that have BOTH long and short CoT for a paired comparison
paired = [ex for ex in all_records if ex.get('long_cot') and ex.get('short_cot')]
sample_size = min(200, len(paired))
rng = random.Random(SEED)
ppl_sample = rng.sample(paired, sample_size)
print(f"PPL sample size: {sample_size}")

In [ ]:
# Compute PPL under base model
free_gpu()
model, tok = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

def avg_loss(text):
    enc = tok(text, return_tensors='pt', truncation=True,
              max_length=MAX_SEQ_LENGTH).to('cuda')
    if enc['input_ids'].shape[1] < 2:
        return None
    with torch.no_grad():
        out = model(**enc, labels=enc['input_ids'])
    return out.loss.item()

long_losses, short_losses = [], []
for ex in tqdm(ppl_sample, desc="PPL"):
    l_text = to_chat_text(ex['prompt'], ex['long_cot'])
    s_text = to_chat_text(ex['prompt'], ex['short_cot'])
    ll = avg_loss(l_text)
    sl = avg_loss(s_text)
    if ll is not None and sl is not None:
        long_losses.append(ll)
        short_losses.append(sl)

import math
long_ppl = math.exp(sum(long_losses) / len(long_losses))
short_ppl = math.exp(sum(short_losses) / len(short_losses))

print(f"\nBase model PPL on long-CoT:   {long_ppl:.3f}")
print(f"Base model PPL on short-CoT:  {short_ppl:.3f}")
print(f"Δ (long − short):             {long_ppl - short_ppl:+.3f}")

ppl_results = {
    'model': BASE_MODEL,
    'n_samples': len(long_losses),
    'long_cot_ppl': long_ppl,
    'short_cot_ppl': short_ppl,
    'delta': long_ppl - short_ppl,
}
with open(f"{OUTPUT_ROOT}/eval_results/perplexity_base.json", 'w') as f:
    json.dump(ppl_results, f, indent=2)

del model, tok
free_gpu()

## 9. Results summary

A single table consolidating all conditions, both benchmarks, and the paper's claims (∆_Long, generalization gap).

In [ ]:
import pandas as pd

def load(label, bench):
    path = f"{OUTPUT_ROOT}/eval_results/{label}__{bench}.json"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

conditions = ['base', 'long_only', 'short_only', 'static_mix_alpha02', 'curriculum_final']
rows = []
for c in conditions:
    he = load(c, 'humaneval')
    mb = load(c, 'mbpp_test')
    rows.append({
        'condition': c,
        'HumanEval Pass@1 (%)': f"{he['pass@1']*100:.2f}" if he else 'n/a',
        'HumanEval 95% CI':    f"[{he['ci95'][0]*100:.2f}, {he['ci95'][1]*100:.2f}]" if he else 'n/a',
        'MBPP-test Pass@1 (%)': f"{mb['pass@1']*100:.2f}" if mb else 'n/a',
        'MBPP-test 95% CI':    f"[{mb['ci95'][0]*100:.2f}, {mb['ci95'][1]*100:.2f}]" if mb else 'n/a',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(f"{OUTPUT_ROOT}/eval_results/summary.csv", index=False)

In [ ]:
# Compute the key contrasts the thesis hinges on
def get(label, bench):
    r = load(label, bench)
    return r['pass@1'] if r else None

print("=== Key contrasts ===\n")

# Paper Takeaway 1 replicated for code: ∆_Long = P_Long − P_Short
for bench in ['humaneval', 'mbpp_test']:
    p_long  = get('long_only', bench)
    p_short = get('short_only', bench)
    if p_long is not None and p_short is not None:
        print(f"∆_Long ({bench:11s}):           {(p_long - p_short)*100:+.2f} points  "
              f"(long={p_long*100:.2f}%, short={p_short*100:.2f}%)")
        print(f"  Negative ∆_Long → small-model learnability gap holds for code")

print()

# Curriculum vs static mix — your novel contribution
for bench in ['humaneval', 'mbpp_test']:
    p_curr = get('curriculum_final', bench)
    p_mix  = get('static_mix_alpha02', bench)
    if p_curr is not None and p_mix is not None:
        print(f"Curriculum − StaticMix ({bench:11s}): {(p_curr - p_mix)*100:+.2f} points  "
              f"(curr={p_curr*100:.2f}%, mix={p_mix*100:.2f}%)")

print()

# In-domain vs OOD generalization gap (the train/test coupling fix in action)
for c in conditions:
    p_he = get(c, 'humaneval')
    p_mb = get(c, 'mbpp_test')
    if p_he is not None and p_mb is not None:
        print(f"  {c:25s}  MBPP={p_mb*100:5.2f}%  HE={p_he*100:5.2f}%  "
              f"gap={(p_mb - p_he)*100:+5.2f}")

In [ ]:
# Plot curriculum trajectory if available
import matplotlib.pyplot as plt

traj_path = f"{OUTPUT_ROOT}/eval_results/curriculum_trajectory.json"
if os.path.exists(traj_path):
    with open(traj_path) as f:
        traj = json.load(f)

    stages = [s['stage'] for s in traj]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(stages, [s['mbpp_test_pass1']*100 for s in traj], 'o-', label='MBPP-test (in-domain)')
    ax.plot(stages, [s['humaneval_pass1']*100 for s in traj], 's-', label='HumanEval (OOD)')
    ax.set_xlabel('Curriculum stage')
    ax.set_ylabel('Pass@1 (%)')
    ax.set_title(f'Curriculum learning trajectory ({BASE_MODEL.split("/")[-1]})')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_ROOT}/eval_results/curriculum_trajectory.png", dpi=120)
    plt.show()

## 10. Notes for the write-up

- **Train/test domain coupling is now addressed:** MBPP held-out test (task_ids 11–510) gives in-domain generalization; HumanEval gives OOD transfer. The `MBPP − HumanEval` gap quantifies how much improvement is distribution-bound.
- **All comparisons share the same base model, seed, decoding strategy, and chat template** — differences in Pass@1 reflect training data, not infrastructure.
- **The static Mix-Long (α=0.2) baseline is the one to beat** — without that baseline you can only claim "mixing helps in code", not "curriculum mixing helps".
- **Perplexity figure** in §8 supports the distribution-mismatch story (Takeaway 4).
- **Per-stage curriculum trajectory** lets you argue for the *progression*, not just the endpoint.
